In [17]:
from bs4 import BeautifulSoup, SoupStrainer
import requests
from langchain_community.document_loaders import WebBaseLoader, SitemapLoader
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from urllib.parse import urljoin, urlparse, urldefrag
import pickle
import os

In [2]:
parse_only = SoupStrainer('div', class_='document')

loader = WebBaseLoader(
    web_paths=("https://docs.manim.community/en/stable/",),
    bs_kwargs=dict(parse_only=parse_only)
)

docs = loader.load()

In [3]:
docs

[Document(page_content='', metadata={'source': 'https://docs.manim.community/en/stable/'})]

In [52]:
def get_all_links(url, base_url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    for link in soup.find_all('a', href=True):
        href = link['href']
        full_url = urljoin(url, href)
        if full_url.startswith(base_url) and full_url != url:
            yield full_url

def crawl_site(start_url, base_url, debug=False):
    visited = set()
    to_visit = [start_url]
    
    while to_visit:
        current_url = to_visit.pop(0)
        current_url, _ = urldefrag(current_url)
        if current_url not in visited and current_url.startswith(base_url):
            if debug:
                print(f"Crawling: {current_url}")
            visited.add(current_url)
            to_visit.extend(link for link in get_all_links(current_url, base_url) if link not in visited)
    
    return list(visited)

def extract_content_with_sections(html, url):
    soup = BeautifulSoup(html, 'html.parser')
    article = soup.find('article')
    if not article:
        return None
    
    main_content = article.get_text(strip=True)

    sections = []
    for h in article.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6']):
        sections.append({'id': h.get('id', ''), 'title': h.get_text(strip=True)})
    
    return {'url': url, 'content': main_content, 'sections': ' '.join(s['title'] for s in sections)}

def get_page_content(url):
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to fetch {url}, status code: {response.status_code}")
        return None
    return extract_content_with_sections(response.text, url)


In [53]:
def scrape_docs(start_url, base_url=None, verbose=False):
    if base_url is None:
        base_url = f"{urlparse(start_url).scheme}://{urlparse(start_url).netloc}"
    if verbose:
        print(f"Base URL: {base_url}")
    all_urls = crawl_site(start_url, start_url, debug=verbose)

    documents = []
    for url in all_urls:
        content = get_page_content(url)
        if content:
            doc = Document(
                page_content=content['content'],
                metadata={'url': content['url'], 'sections': content['sections']}
            )

            documents.append(doc)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    
    split_docs = text_splitter.split_documents(documents)
    if verbose:
        print(f"Scraped {len(documents)} documents")
        print(f"Split into {len(split_docs)} chunks")
        for doc in documents:
            print(doc.metadata['url'])
            print(doc.page_content[:100])
    return split_docs

In [54]:
docs = scrape_docs("https://docs.manim.community/en/stable/", verbose=True)

Base URL: https://docs.manim.community
Crawling: https://docs.manim.community/en/stable/
Crawling: https://docs.manim.community/en/stable/examples.html
Crawling: https://docs.manim.community/en/stable/installation.html
Crawling: https://docs.manim.community/en/stable/installation/conda.html
Crawling: https://docs.manim.community/en/stable/installation/windows.html
Crawling: https://docs.manim.community/en/stable/installation/macos.html
Crawling: https://docs.manim.community/en/stable/installation/linux.html
Crawling: https://docs.manim.community/en/stable/installation/docker.html
Crawling: https://docs.manim.community/en/stable/installation/jupyter.html
Crawling: https://docs.manim.community/en/stable/tutorials_guides.html
Crawling: https://docs.manim.community/en/stable/tutorials/index.html
Crawling: https://docs.manim.community/en/stable/tutorials/quickstart.html
Crawling: https://docs.manim.community/en/stable/tutorials/output_and_config.html
Crawling: https://docs.manim.community/e

In [38]:
parse_only = SoupStrainer('div', class_='document')


In [49]:
def scrape_manim_docs(start_url):
    # base_url = f"{urlparse(start_url).scheme}://{urlparse(start_url).netloc}"
    all_urls = crawl_site(start_url, start_url)
    # Define SoupStrainer to parse only the main content
    # parse_only = SoupStrainer('div', class_='document')

    # def extract_content(html):
    #     soup = BeautifulSoup(html, 'html.parser')
    #     article = soup.find('article')
    #     return article.get_text(strip=True) if article else None
    
    # loader = WebBaseLoader(
    #     web_paths=all_urls,
    #     bs_kwargs={"parse_only": extract_content},
    # )


    # documents = loader.load()

    documents = []
    for url in all_urls: 
        print(f"Processing URL: {url}")
        response = requests.get(url)        
        soup = BeautifulSoup(response.text, 'html.parser')

        article = soup.find('article')
        
        if article:
            content = article.get_text(strip=True)

            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=1000,
                chunk_overlap=200,
                length_function=len
            )

            chunks = text_splitter.split_text(content)

            for chunk in chunks:
                documents.append(Document(page_content=chunk, metadata={"source": url}))

            print(f"Processed {len(chunks)} chunks from {url}")
        else:
            print(f"No article found in {url}")

    
    
    print(f"Total Documents created: {len(documents)}")
    return documents
    # loader = WebBaseLoader(
    #     web_paths=all_urls,
    #     bs_kwargs={"parse_only": parse_only}
    # )
    # documents = loader.load()
    # return documents

    
    # Split documents
    # text_splitter = RecursiveCharacterTextSplitter(
    #     chunk_size=1000,
    #     chunk_overlap=200,
    #     length_function=len
    # )
    
    # split_docs = text_splitter.split_documents(documents)
    # return split_docs

In [50]:
docs = scrape_manim_docs("https://docs.manim.community/en/stable/tutorials/")

Processing URL: https://docs.manim.community/en/stable/tutorials/output_and_config.html
Processed 11 chunks from https://docs.manim.community/en/stable/tutorials/output_and_config.html
Processing URL: https://docs.manim.community/en/stable/tutorials/quickstart.html
Processed 13 chunks from https://docs.manim.community/en/stable/tutorials/quickstart.html
Processing URL: https://docs.manim.community/en/stable/tutorials/building_blocks.html
Processed 38 chunks from https://docs.manim.community/en/stable/tutorials/building_blocks.html
Processing URL: https://docs.manim.community/en/stable/tutorials/index.html
Processed 1 chunks from https://docs.manim.community/en/stable/tutorials/index.html
Processing URL: https://docs.manim.community/en/stable/tutorials/
Processed 1 chunks from https://docs.manim.community/en/stable/tutorials/
Total Documents created: 64


In [51]:
for i, doc in enumerate(docs[:3]): # Debugging
    print(f"Document {i + 1}:")
    print(f"Content length: {len(doc.page_content)}")
    print(f"Content preview: {doc.page_content[:500]}")  
    print(f"Metadata: {doc.metadata}")
    print("---")

Document 1:
Content length: 955
Content preview: Manim’s Output Settings¶This document will focus on understanding manim’s output files and some of the
main command-line flags available.NoteThis tutorial picks up whereQuickstartleft off, so please
read that document before starting this one.Manim output folders¶At this point, you have just executed the following command.manim-pqlscene.pySquareToCircleLet’s dissect what just happened step by step.  First, this command executes
manim on the filescene.py, which contains our animation code.  Furth
Metadata: {'source': 'https://docs.manim.community/en/stable/tutorials/output_and_config.html'}
---
Document 2:
Content length: 963
Content preview: quality.After the video is rendered, you will see that manim has generated some new
files and the project folder will look as follows.project/
├─scene.py
└─media├─videos|└─scene|└─480p15|├─SquareToCircle.mp4|└─partial_movie_files├─text└─TexThere are quite a few new files.  The main output is inmedia/

In [55]:
def save_documents(documents, filename):
    with open(filename, 'wb') as f:
        pickle.dump(documents, f)
    print(f"Documents saved to {filename}")


save_documents(docs, "manim_docs.pkl")

Documents saved to manim_docs.pkl


In [56]:
def load_documents(filename):
    with open(filename, 'rb') as f:
        documents = pickle.load(f)
    print(f"Documents loaded from {filename}")
    return documents

loaded_docs = load_documents("manim_docs.pkl")

Documents loaded from manim_docs.pkl


In [57]:
loaded_docs[:3]

[Document(page_content='PhaseFlow¶Qualified name:manim.animation.movement.PhaseFlowclassPhaseFlow(mobject=None,*args,use_override=True,**kwargs)[source]¶Bases:AnimationMethodsinterpolate_mobjectInterpolates the mobject of theAnimationbased on alpha value.Parameters:function(Callable[[np.ndarray],np.ndarray])mobject(Mobject)virtual_time(float)suspend_mobject_updating(bool)rate_func(Callable[[float],float])interpolate_mobject(alpha)[source]¶Interpolates the mobject of theAnimationbased on alpha value.Parameters:alpha(float) – A float between 0 and 1 expressing the ratio to which the animation\nis completed. For example, alpha-values of 0, 0.5, and 1 correspond\nto the animation being completed 0%, 50%, and 100%, respectively.Return type:None', metadata={'url': 'https://docs.manim.community/en/stable/reference/manim.animation.movement.PhaseFlow.html', 'sections': 'PhaseFlow¶'}),
 Document(page_content='Union¶Qualified name:manim.mobject.geometry.boolean\\_ops.UnionclassUnion(*vmobjects,**

In [58]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain_google_vertexai import (
    VertexAI,
    ChatVertexAI,
    VertexAIEmbeddings,
    VectorSearchVectorStore
)
model_name = "sentence-transformers/all-MiniLM-L6-v2"

hf_embeddings = HuggingFaceEmbeddings(model_name=model_name)

vectorstore = Chroma.from_documents(documents=loaded_docs, embedding=hf_embeddings)


/Users/jamessong/Desktop/an-gen/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [59]:
vectorstore.persist()

In [60]:
query = "Neural Networks"
docs = vectorstore.similarity_search(query, k=5)

In [61]:
docs

[Document(page_content='def construct(self):\n        t2cindices = Text(\'Hello\', t2c={\'[1:-1]\': BLUE}).move_to(LEFT)\n        t2cwords = Text(\'World\',t2c={\'rl\':RED}).next_to(t2cindices, RIGHT)\n        self.add(t2cindices, t2cwords)If you want to avoid problems when using colors (due to ligatures), consider usingMarkupText.Using Gradients¶You can add a gradient usinggradient. The value must\nbe an iterable of any length:Example: GradientExample¶frommanimimport*classGradientExample(Scene):defconstruct(self):t=Text("Hello",gradient=(RED,BLUE,GREEN),font_size=96)self.add(t)class GradientExample(Scene):\n    def construct(self):\n        t = Text("Hello", gradient=(RED, BLUE, GREEN), font_size=96)\n        self.add(t)You can also uset2gfor gradients with specific\ncharacters of the text. It shares a similar syntax tothe', metadata={'sections': 'Rendering Text and Formulas¶ Text Without LaTeX¶ Working withText¶ Using Fonts¶ Setting Slant and Weight¶ Using Colors¶ Using Gradients¶ Se